In [ ]:
import feedparser
import yfinance as yf
import google.generativeai as genai

# =========================
# CONFIG GEMINI
# =========================
genai.configure(api_key="")

model = genai.GenerativeModel("gemini-2.5-flash-lite")

# =========================
# MCP SERVER (SIMPLIFIED)
# =========================
class MCPServer:
    def __init__(self):
        self.tools = {}

    def tool(self, func):
        self.tools[func.__name__] = func
        return func

    def run_tool(self, name, *args):
        return self.tools[name](*args)


mcp = MCPServer()

# =========================
# TOOLS (MCP RESOURCES)
# =========================

@mcp.tool
def get_news(topic: str):
    url = f"https://news.google.com/rss/search?q={topic}"
    feed = feedparser.parse(url)
    return "\n".join([e.title for e in feed.entries[:8]])


@mcp.tool
def get_stock_data(ticker: str):
    stock = yf.Ticker(ticker)
    hist = stock.history(period="5d")

    latest = hist.iloc[-1]
    prev = hist.iloc[-2]

    change = latest["Close"] - prev["Close"]
    pct = (change / prev["Close"]) * 100

    return f"Change: {change:.2f} ({pct:.2f}%)"


# =========================
# PROMPT BUILDER (MCP PROMPT)
# =========================

def build_prompt(news, stock):
    return f"""
You are a financial analyst.

Stock data:
{stock}

News:
{news}

Step 1: Decide if news is relevant
Step 2: Identify causal reason
Step 3: Output ONE sentence explanation
"""


# =========================
# GEMINI WRAPPER
# =========================

def llm(prompt: str):
    return model.generate_content(prompt).text


# =========================
# MCP AGENT LOOP (IMPORTANT)
# =========================

def analyze_stock(ticker: str):
    print("\n======================")
    print("MCP AGENT RUNNING")
    print("======================\n")

    # STEP 1 — CALL TOOLS
    news = mcp.run_tool("get_news", ticker)
    stock = mcp.run_tool("get_stock_data", ticker)

    print("📊 Stock Data:")
    print(stock)

    print("\n📰 News:")
    print(news)

    # STEP 2 — BUILD PROMPT
    prompt = build_prompt(news, stock)

    # STEP 3 — CALL AI
    result = llm(prompt)

    print("\n🧠 FINAL REASON:")
    print(result)


# =========================
# RUN
# =========================

analyze_stock("AAPL")